# 💊 PHARMAWATCH
## Plataforma de farmacovigilancia con perspectiva de sexo

Este notebook permite analizar señales de riesgo diferencial en medicamentos comparando patrones de efectos adversos entre mujeres y hombres a partir de datos reales de la FDA.

**Fuente de datos:** FDA FAERS via openFDA API  
**Referencia farmacológica:** RxNorm API (NIH)  
**Algoritmos:** PRR (Proportional Reporting Ratio) y ROR (Reporting Odds Ratio)

---
▶️ **Ejecuta las celdas en orden. En cada celda interactiva introduce los datos que se te piden.**

## 1. Instalación e imports

In [1]:
!pip install pandas matplotlib requests ipywidgets -q
print('✓ Dependencias instaladas')

✓ Dependencias instaladas


In [3]:
import sys
import os

# Añadir el directorio del proyecto al path para poder importar pharmawatch
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from pharmawatch.loader import FAERSLoader
from pharmawatch.analyzer import SexStratifiedAnalysis
from pharmawatch.reference_finder import ReferenceFinder
from pharmawatch.visualizer import SignalVisualizer
from pharmawatch.exceptions import InsufficientDataError, SexFieldMissingError, InvalidDrugNameError

# Estado global del análisis
state = {
    'drugs_to_analyze': [],
    'reference_drugs': [],
    'all_drugs': [],
    'full_df': None,
    'results': None,
    'disease_label': ''
}

print('✓ pharmawatch importado correctamente')

✓ pharmawatch importado correctamente


## 2. ¿Qué fármacos quieres analizar?

Introduce uno o varios fármacos separados por comas **(en inglés)**. Ejemplo: `ibuprofen, aspirin`

In [4]:
drug_input = widgets.Text(
    placeholder='ej: ibuprofen, aspirin',
    description='Fármacos:',
    layout=widgets.Layout(width='500px')
)
btn_drugs = widgets.Button(description='Confirmar', button_style='primary')
out_drugs = widgets.Output()

def on_confirm_drugs(b):
    with out_drugs:
        clear_output()
        raw = drug_input.value.strip()
        if not raw:
            print('⚠️  Introduce al menos un fármaco.')
            return
        drugs = [d.strip().lower() for d in raw.split(',') if d.strip()]
        state['drugs_to_analyze'] = drugs
        print(f'✓ Fármacos seleccionados: {", ".join(drugs)}')

btn_drugs.on_click(on_confirm_drugs)
display(widgets.VBox([drug_input, btn_drugs, out_drugs]))

## 3. ¿Quieres comparar con fármacos similares?

Puedes buscar fármacos de referencia automáticamente por **enfermedad/síntoma** o **mecanismo de acción**, o bien saltar este paso.

In [5]:
want_ref = widgets.ToggleButtons(
    options=['Sí, buscar similares', 'No, solo analizar mis fármacos'],
    description='',
    button_style=''
)
criterion = widgets.RadioButtons(
    options=['Enfermedad / síntoma que tratan', 'Mecanismo de acción / clase farmacológica'],
    description='Criterio:',
    layout=widgets.Layout(width='500px')
)
top_n_widget = widgets.IntSlider(
    value=5, min=1, max=50, step=1,
    description='Top N:',
    style={'description_width': 'initial'}
)
btn_ref = widgets.Button(description='Buscar fármacos similares', button_style='info')
btn_skip = widgets.Button(description='Saltar este paso', button_style='warning')
out_ref = widgets.Output()
classes_dropdown = widgets.Dropdown(description='Clase:', layout=widgets.Layout(width='500px'))
btn_confirm_class = widgets.Button(description='Confirmar clase', button_style='primary')
out_classes = widgets.Output()

def on_skip_ref(b):
    with out_ref:
        clear_output()
        state['reference_drugs'] = []
        state['disease_label'] = ''
        print('✓ Análisis sin fármacos de referencia.')

def on_search_ref(b):
    with out_ref:
        clear_output()
        if not state['drugs_to_analyze']:
            print('⚠️  Primero confirma los fármacos en el paso 2.')
            return
        base_drug = state['drugs_to_analyze'][0]
        print(f'Buscando clases RxNorm para "{base_drug}"...')
        try:
            finder = ReferenceFinder(base_drug)
            finder.fetch_classes()
            state['_finder'] = finder
        except Exception as e:
            print(f'❌ Error: {e}')
            return

        use_disease = 'Enfermedad' in criterion.value
        classes = finder.get_disease_classes() if use_disease else finder.get_moa_classes()
        if not classes:
            print('⚠️  No se encontraron clases disponibles.')
            return

        state['_classes'] = classes
        classes_dropdown.options = [(c['class_name'], i) for i, c in enumerate(classes)]
        print(f'✓ {len(classes)} clases encontradas. Selecciona una:')
        display(widgets.VBox([classes_dropdown, top_n_widget, btn_confirm_class, out_classes]))

def on_confirm_class(b):
    with out_classes:
        clear_output()
        idx = classes_dropdown.value
        selected = state['_classes'][idx]
        top_n = top_n_widget.value
        print(f'Buscando top {top_n} fármacos similares por "{selected["class_name"]}"...')
        try:
            refs = state['_finder'].get_similar_drugs(
                class_id=selected['class_id'],
                rela=selected['rela'],
                top_n=top_n
            )
            refs = [d for d in refs if d not in state['drugs_to_analyze']]
            state['reference_drugs'] = refs
            state['disease_label'] = f'Indicación: {selected["class_name"]}'
            print(f'✓ {len(refs)} fármacos de referencia encontrados:')
            for d in refs:
                print(f'   - {d}')
        except Exception as e:
            print(f'❌ Error: {e}')

btn_ref.on_click(on_search_ref)
btn_skip.on_click(on_skip_ref)
btn_confirm_class.on_click(on_confirm_class)
display(widgets.VBox([want_ref, criterion, widgets.HBox([btn_ref, btn_skip]), out_ref]))

## 4. Descarga de datos — openFDA API

In [6]:
max_records_widget = widgets.IntSlider(
    value=300, min=50, max=1000, step=50,
    description='Registros por fármaco:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
btn_load = widgets.Button(description='Descargar datos', button_style='success')
out_load = widgets.Output()

def on_load(b):
    with out_load:
        clear_output()
        if not state['drugs_to_analyze']:
            print('⚠️  Confirma los fármacos en el paso 2.')
            return

        all_drugs = list(dict.fromkeys(state['drugs_to_analyze'] + state['reference_drugs']))
        state['all_drugs'] = all_drugs
        print(f'Descargando datos para {len(all_drugs)} fármaco(s)...\n')

        dfs = []
        for drug in all_drugs:
            print(f'  · {drug:<25}', end=' ')
            try:
                loader = FAERSLoader(drug_name=drug, max_records=max_records_widget.value)
                df = loader.load()
                dfs.append(df)
                print(f'{len(df):>5} registros  (F:{(df["sex"]=="F").sum()} / M:{(df["sex"]=="M").sum()})')
            except InvalidDrugNameError as e:
                print(f'NOMBRE INVÁLIDO — {e.message}')
            except InsufficientDataError as e:
                print(f'POCOS DATOS — {e.message}')
            except Exception as e:
                print(f'ERROR — {e}')

        if not dfs:
            print('❌ No se pudo cargar ningún fármaco.')
            return

        state['full_df'] = pd.concat(dfs, ignore_index=True)
        print(f'\n✓ Total registros cargados: {len(state["full_df"])}')
        display(state['full_df'].groupby(['drug_name','sex']).size().unstack(fill_value=0))

btn_load.on_click(on_load)
display(widgets.VBox([max_records_widget, btn_load, out_load]))

## 5. Análisis PRR + ROR estratificado por sexo

In [7]:
btn_analyze = widgets.Button(description='Ejecutar análisis', button_style='primary')
out_analyze = widgets.Output()

def on_analyze(b):
    with out_analyze:
        clear_output()
        if state['full_df'] is None:
            print('⚠️  Primero descarga los datos en el paso 4.')
            return

        full_df = state['full_df']
        all_drugs = state['all_drugs']
        drugs_to_analyze = state['drugs_to_analyze']

        has_reference = len(state['reference_drugs']) > 0

        if not has_reference:
            print('Modo sin comparativa — top reacciones por fármaco:\n')
            for drug in drugs_to_analyze:
                drug_df = full_df[full_df['drug_name'] == drug]
                print(f'\n=== {drug.upper()} ===')
                for sex, label in [('F','MUJERES'),('M','HOMBRES')]:
                    top = drug_df[drug_df['sex']==sex]['reaction'].value_counts().head(10)
                    print(f'\n{label}:')
                    display(top.to_frame())
            state['results'] = pd.DataFrame()
            return

        valid_drugs = [d for d in all_drugs if len(full_df[full_df['drug_name']==d]) >= 3]
        skipped = set(all_drugs) - set(valid_drugs)
        if skipped:
            print(f'⚠️  Fármacos omitidos por datos insuficientes: {", ".join(skipped)}\n')
        for d in drugs_to_analyze:
            if d not in valid_drugs:
                valid_drugs.append(d)

        print('Ejecutando análisis PRR + ROR...')
        try:
            analysis = SexStratifiedAnalysis(
                df=full_df,
                drug_filter=valid_drugs,
                prr_threshold=2.0,
                ci_level=0.95,
                min_records=3
            )
            results = analysis.run()
            state['results'] = results
            signals = results[results['is_signal']]
            print(f'✓ Análisis completado: {results["is_signal"].sum()} señales detectadas\n')

            for drug in drugs_to_analyze:
                drug_signals = signals[signals['drug_name']==drug]
                print(f'\n=== {drug.upper()} ===')
                for sex, label in [('F','MUJERES'),('M','HOMBRES')]:
                    sex_s = drug_signals[drug_signals['sex']==sex].sort_values('prr', ascending=False)
                    print(f'\n{label} — {len(sex_s)} señales:')
                    if len(sex_s) > 0:
                        display(sex_s[['reaction','n_cases','prr','ror','ci_lower','ci_upper']].head(10).reset_index(drop=True))
                    else:
                        print('  (ninguna señal por encima del umbral)')
        except Exception as e:
            print(f'❌ Error: {e}')

btn_analyze.on_click(on_analyze)
display(widgets.VBox([btn_analyze, out_analyze]))

## 6. Visualización de resultados

In [8]:
btn_viz = widgets.Button(description='Generar gráficos', button_style='success')
out_viz = widgets.Output()

def on_viz(b):
    with out_viz:
        clear_output()
        if state['results'] is None or state['results'].empty:
            print('⚠️  Primero ejecuta el análisis en el paso 5.')
            return
        viz = SignalVisualizer(state['results'])
        print('Generando gráfico 1: comparativa entre fármacos seleccionados...')
        viz.plot_user_drugs_comparison(drugs_to_analyze=state['drugs_to_analyze'])
        plt.show()
        print('Generando gráfico 2: ranking de seguridad de todos los fármacos...')
        viz.plot_drug_ranking(disease_label=state['disease_label'])
        plt.show()

btn_viz.on_click(on_viz)
display(widgets.VBox([btn_viz, out_viz]))

## 7. Exploración libre de datos

In [9]:
# Distribución de reportes por fármaco y sexo
if state['full_df'] is not None:
    display(state['full_df'].groupby(['drug_name','sex']).size().unstack(fill_value=0))

sex,F,M
drug_name,,
acetaminophen,797,374
aspirin,461,505
benzocaine,793,336
ibuprofen,838,374
tetracaine,737,408


In [10]:
# GroupBy: top reacciones por fármaco y sexo
if state['full_df'] is not None:
    grouped = (
        state['full_df']
        .groupby(['drug_name','sex','reaction'])['report_id']
        .nunique()
        .reset_index(name='n_reports')
        .sort_values('n_reports', ascending=False)
    )
    display(grouped.head(15))

,drug_name,sex,reaction,n_reports
1223,benzocaine,F,drug ineffective,36
408,acetaminophen,F,toxicity to various agents,27
638,acetaminophen,M,toxicity to various agents,24
2083,ibuprofen,F,pain,22
1602,benzocaine,M,drug ineffective,20
1356,benzocaine,F,methaemoglobinaemia,20
954,aspirin,M,fatigue,19
748,aspirin,F,haemoglobin decreased,17
308,acetaminophen,F,overdose,17
1222,benzocaine,F,drug hypersensitivity,17


In [11]:
# Señales exclusivas por sexo
if state['results'] is not None and not state['results'].empty:
    signals = state['results'][state['results']['is_signal']]
    for drug in state['drugs_to_analyze']:
        drug_s = signals[signals['drug_name']==drug]
        only_f = set(drug_s[drug_s['sex']=='F']['reaction']) - set(drug_s[drug_s['sex']=='M']['reaction'])
        only_m = set(drug_s[drug_s['sex']=='M']['reaction']) - set(drug_s[drug_s['sex']=='F']['reaction'])
        print(f'\n{drug.upper()}')
        print(f'  Señales exclusivas en mujeres: {", ".join(list(only_f)[:5]) or "ninguna"}')
        print(f'  Señales exclusivas en hombres: {", ".join(list(only_m)[:5]) or "ninguna"}')


IBUPROFEN
  Señales exclusivas en mujeres: emotional distress, pulmonary embolism, injury, renal impairment, chronic obstructive pulmonary disease
  Señales exclusivas en hombres: skin exfoliation, feeling abnormal, haemoptysis, nasopharyngitis, stevens-johnson syndrome

ASPIRIN
  Señales exclusivas en mujeres: bone pain, loss of consciousness, blood count abnormal, platelet count increased, flatulence
  Señales exclusivas en hombres: hyperhidrosis, cough, weight decreased, haematocrit decreased, fatigue
